# Malaga — Listings: Exploration & Cleaning
**Input :** `../../Data/raw/Malaga/listings.csv.gz`  
**Output:** `../../Data/interim/malaga_listings_clean.parquet`

Runs the shared `clean_listings()` pipeline, applies city-specific outlier
decisions, and exports a clean Parquet file consumed by the analysis notebook.

In [ ]:
# !pip install pyarrow wordcloud textblob  # run once if needed

In [ ]:
import sys
sys.path.insert(0, "../..")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import f_oneway, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from airbnb_iip.data.cleaning import (
    clean_listings, missing_report,
    standardize_property_type, rate_bucket,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 200)

## 1 · Load raw data

In [ ]:
l_df_raw = pd.read_csv(
    "../../Data/raw/Malaga/listings.csv.gz",
    compression="gzip",
    low_memory=False,
)
print(f"Shape  : {l_df_raw.shape}")
print(f"Memory : {l_df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2 · Initial exploration

In [ ]:
l_df_raw.info()

In [ ]:
display(l_df_raw.head())
display(l_df_raw.tail())

In [ ]:
l_df_raw.describe().T

In [ ]:
mr_raw = missing_report(l_df_raw)
mr_raw[mr_raw.missing_count > 0]["missing_percent"].plot(
    kind="bar", figsize=(14, 5), title="% Missing — raw"
)
display(mr_raw.head(20))

In [ ]:
# Unique-value counts — categorical columns (skip long free-text)
_skip = {"description", "host_about", "neighborhood_overview", "amenities", "name"}
for col in l_df_raw.select_dtypes("object").columns.difference(_skip):
    print(f"\n{col}  ({l_df_raw[col].nunique()} unique)")
    print(l_df_raw[col].value_counts().head(6).to_string())

In [ ]:
# Numeric ranges — flag obvious anomalies before cleaning
for col in l_df_raw.select_dtypes(include=["int64", "float64"]).columns:
    print(f"{col}: min={l_df_raw[col].min()}, max={l_df_raw[col].max()}")

In [ ]:
# Duplicate check
print("Full duplicates           :", l_df_raw.duplicated().sum())
print("Duplicate listing ids     :", l_df_raw.duplicated(subset=['id']).sum())
print("Duplicate (name+host+coords):",
      l_df_raw.duplicated(subset=['name','host_id','latitude','longitude']).sum())

## 3 · Cleaning pipeline

`clean_listings()` handles (in order):
1. Drop `calendar_updated`, `neighbourhood` (always useless)
2. Deduplicate ignoring `id` / `listing_url`
3. Parse dates, prices, rates, booleans
4. Normalise multi-host name separators → `&`
5. Reconcile `bathrooms` / `bathrooms_text` (fixes the `fillna("mode")` bug from legacy code)
6. Hierarchical imputation — host fields → price → beds/bedrooms
7. Review-score zeros / sentinel dates for never-reviewed listings
8. Temporal features: `host_tenure_years`, `days_since_*`, `review_span_years`
9. Engineered features: `property_type_std`, rate categories, `price_cat` (city-adaptive quartiles), `description_length`
10. Drop URL / thumbnail columns

In [ ]:
l_df = clean_listings(l_df_raw)
print(f"After clean_listings(): {l_df.shape}")
l_df.info()

### City-specific decisions

Document outlier thresholds and the EDA evidence that justifies them.

In [ ]:
# TODO: after inspecting the distributions above, add city-specific
# outlier filters here.  Example:
#   l_df = l_df[l_df['beds'] <= 20]
print(f'After city-specific outlier removal: {l_df.shape}')

In [ ]:
mr_clean = missing_report(l_df)
remaining = mr_clean[mr_clean.missing_count > 0]
if remaining.empty:
    print("No missing values remaining ✓")
else:
    display(remaining)
    remaining["missing_percent"].plot(
        kind="bar", figsize=(12, 4), title="% Missing — after cleaning"
    )

In [ ]:
# Confirm final dtypes and feature set
l_df.info()
print("\nEngineered columns added:")
new_cols = set(l_df.columns) - set(l_df_raw.columns)
print(sorted(new_cols))

## 4 · Export cleaned parquet

Output: `../../Data/interim/malaga_listings_clean.parquet`

In [ ]:
import pathlib

out_path = pathlib.Path("../../Data/interim/malaga_listings_clean.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)

l_df.to_parquet(out_path, index=False, engine="pyarrow")

print(f"Saved {l_df.shape[0]:,} rows × {l_df.shape[1]} cols → {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")